In [2]:
import os
from glob import glob
import numpy as np
import pandas as pd
from tqdm import tqdm

# 데이터 경로 설정
data_path = "../../../Downloads/archive"
data_deaths_files = sorted(glob(os.path.join(data_path, 'deaths/*.csv')))

# 저장 디렉토리 생성
output_dir = os.path.join(data_path, "processed_npy")
os.makedirs(output_dir, exist_ok=True)

# 파일 처리
# for idx, file_path in enumerate(tqdm(data_deaths_files, desc="Processing death logs")):
#     try:
#         df = pd.read_csv(file_path)
#         df = df.dropna()

#         # (0.0, 0.0) 위치 제거
#         df = df[~((df['victim_position_x'] == 0.0) & (df['victim_position_y'] == 0.0))]
#         df = df[~((df['killer_position_x'] == 0.0) & (df['killer_position_y'] == 0.0))]

#         # ERANGEL 맵 필터링
#         df = df[df['map'] == 'ERANGEL'].copy()
#         if df.empty:
#             continue

#         # 상대 위치 계산
#         df['delta_x'] = df['killer_position_x'] - df['victim_position_x']
#         df['delta_y'] = df['killer_position_y'] - df['victim_position_y']

#         # 입력 / 출력 추출
#         input_data = df[['time', 'victim_position_x', 'victim_position_y']].to_numpy()
#         target_data = df[['delta_x', 'delta_y']].to_numpy()

#         if len(input_data) == 0:
#             continue

#         # 저장 (.npz 파일 하나에 input과 target 모두 저장)
#         save_path = os.path.join(output_dir, f"mdn_data_{idx:03d}.npz")
#         np.savez_compressed(save_path, input=input_data, target=target_data)

#     except Exception as e:
#         print(f"Error processing file {file_path}: {e}")


In [3]:
for idx, file_path in enumerate(tqdm(data_deaths_files, desc="Processing death logs")):
    try:
        df = pd.read_csv(file_path)
        df = df.dropna()

        # (0.0, 0.0) 위치 제거
        df = df[~((df['victim_position_x'] == 0.0) & (df['victim_position_y'] == 0.0))]
        df = df[~((df['killer_position_x'] == 0.0) & (df['killer_position_y'] == 0.0))]

        # ERANGEL 맵 필터링
        df = df[df['map'] == 'ERANGEL'].copy()
        if df.empty:
            continue

        # 거리 계산
        dx = df['killer_position_x'] - df['victim_position_x']
        dy = df['killer_position_y'] - df['victim_position_y']
        distance = np.sqrt(dx**2 + dy**2)

        # 거리 필터링 (100000 이하 전체)
        within_100k_mask = distance <= 100000
        df_all = df[within_100k_mask]
        if df_all.empty:
            continue

        # 상대 위치 계산
        df_all['delta_x'] = df_all['killer_position_x'] - df_all['victim_position_x']
        df_all['delta_y'] = df_all['killer_position_y'] - df_all['victim_position_y']

        input_data_all = df_all[['time', 'victim_position_x', 'victim_position_y']].to_numpy()
        target_data_all = df_all[['delta_x', 'delta_y']].to_numpy()

        if len(input_data_all) > 0:
            save_path_all = os.path.join(output_dir, f"mdn_data_all_{idx:03d}.npz")
            np.savez_compressed(save_path_all, input=input_data_all, target=target_data_all)

        # 거리 필터링 (80000 ~ 100000)
        far_mask = (distance >= 60000) & (distance <= 100000)
        df_far = df[far_mask]

        if not df_far.empty:
            df_far['delta_x'] = df_far['killer_position_x'] - df_far['victim_position_x']
            df_far['delta_y'] = df_far['killer_position_y'] - df_far['victim_position_y']

            input_data_far = df_far[['time', 'victim_position_x', 'victim_position_y']].to_numpy()
            target_data_far = df_far[['delta_x', 'delta_y']].to_numpy()

            save_path_far = os.path.join(output_dir, f"mdn_data_mid_{idx:03d}.npz")
            np.savez_compressed(save_path_far, input=input_data_far, target=target_data_far)
            
        # far_mask = (distance >= 80000) & (distance <= 100000)
        # df_far = df[far_mask]

        # if not df_far.empty:
        #     df_far['delta_x'] = df_far['killer_position_x'] - df_far['victim_position_x']
        #     df_far['delta_y'] = df_far['killer_position_y'] - df_far['victim_position_y']

        #     input_data_far = df_far[['time', 'victim_position_x', 'victim_position_y']].to_numpy()
        #     target_data_far = df_far[['delta_x', 'delta_y']].to_numpy()

        #     save_path_far = os.path.join(output_dir, f"mdn_data_far_{idx:03d}.npz")
        #     np.savez_compressed(save_path_far, input=input_data_far, target=target_data_far)

    except Exception as e:
        print(f"Error processing file {file_path}: {e}")

C:\Users\darke\AppData\Local\Temp\ipykernel_29684\3861722542.py:27: SettingWithCopyWarning:                        | 0/5 [00:00<?, ?it/s]
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_all['delta_x'] = df_all['killer_position_x'] - df_all['victim_position_x']
C:\Users\darke\AppData\Local\Temp\ipykernel_29684\3861722542.py:28: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_all['delta_y'] = df_all['killer_position_y'] - df_all['victim_position_y']
C:\Users\darke\AppData\Local\Temp\ipykernel_29684\3861722542.py:42: SettingWithCopy